# Project 09: Small Language Model from Scratch (Character LM)

Train a compact character-level language model on Tiny Shakespeare and track
gradients and warnings with NeuroGebra observability components.


## Dataset Source and Download Instructions

- Dataset: Tiny Shakespeare corpus
- Official source used by notebook: https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
- License: follow source repository terms
- Auto-download in this notebook: direct URL download
- Manual fallback:
  1. Download the file from the source URL.
  2. Save it as `./data/tiny_shakespeare.txt`.
  3. Re-run data loading cells.


In [ ]:
%pip -q install "neurogebra[logging]" torch matplotlib


In [ ]:
import random
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from neurogebra.logging.epoch_summary import EpochSummarizer
from neurogebra.logging.health_warnings import AutoHealthWarnings
from neurogebra.logging.logger import LogLevel, TrainingLogger

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
data_dir = Path("./data")
data_dir.mkdir(parents=True, exist_ok=True)
text_path = data_dir / "tiny_shakespeare.txt"

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not text_path.exists():
    urlretrieve(url, text_path)

text = text_path.read_text(encoding="utf-8")
print("Corpus length:", len(text))

chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
print("Vocab size:", vocab_size)

encoded = torch.tensor([stoi[c] for c in text], dtype=torch.long)
split = int(0.9 * len(encoded))
train_ids = encoded[:split]
val_ids = encoded[split:]


In [ ]:
context_len = 64
batch_size = 64

def get_batch(data_ids):
    ix = torch.randint(0, len(data_ids) - context_len - 1, (batch_size,))
    x = torch.stack([data_ids[i:i+context_len] for i in ix])
    y = torch.stack([data_ids[i+1:i+context_len+1] for i in ix])
    return x.to(device), y.to(device)

class CharLM(nn.Module):
    def __init__(self, vocab, emb=128, hidden=256):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb)
        self.rnn = nn.GRU(emb, hidden, batch_first=True)
        self.head = nn.Linear(hidden, vocab)

    def forward(self, x, hidden=None):
        e = self.emb(x)
        out, hidden = self.rnn(e, hidden)
        logits = self.head(out)
        return logits, hidden

model = CharLM(vocab_size).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

logger = TrainingLogger(level=LogLevel.DETAILED)
warnings_engine = AutoHealthWarnings()
summarizer = EpochSummarizer()


In [ ]:
def estimate_val_loss(eval_steps=30):
    model.eval()
    losses = []
    with torch.no_grad():
        for _ in range(eval_steps):
            xb, yb = get_batch(val_ids)
            logits, _ = model(xb)
            loss = criterion(logits.reshape(-1, vocab_size), yb.reshape(-1))
            losses.append(float(loss.item()))
    model.train()
    return float(np.mean(losses))

max_steps = 400
eval_every = 100

train_curve = []
val_curve = []

logger.on_train_start(total_epochs=max_steps // eval_every, total_samples=len(train_ids), batch_size=batch_size)

for step in range(1, max_steps + 1):
    xb, yb = get_batch(train_ids)
    optimizer.zero_grad()
    logits, _ = model(xb)
    loss = criterion(logits.reshape(-1, vocab_size), yb.reshape(-1))
    loss.backward()

    grad_norms = {}
    for n, p in model.named_parameters():
        if p.grad is not None:
            gn = float(p.grad.data.norm().item())
            grad_norms[n] = gn
            logger.on_gradient_computed(n, gn)

    optimizer.step()
    step_loss = float(loss.item())

    logger.on_batch_end(step, epoch=step // eval_every, loss=step_loss)
    summarizer.record_batch(
        epoch=step // eval_every,
        metrics={"loss": step_loss},
        gradient_norms=grad_norms,
    )

    alerts = warnings_engine.check_batch(
        epoch=step // eval_every,
        batch=step,
        loss=step_loss,
        gradient_norms=grad_norms,
    )
    for alert in alerts:
        logger.on_health_check(
            check_name=alert.rule_name,
            severity=alert.severity,
            message=alert.message,
            recommendations=alert.recommendations,
        )

    if step % eval_every == 0:
        val_loss = estimate_val_loss()
        train_curve.append(step_loss)
        val_curve.append(val_loss)
        epoch_idx = step // eval_every
        print(f"Step {step}: train_loss={step_loss:.4f}, val_loss={val_loss:.4f}")
        print(summarizer.finalize_epoch(epoch_idx).format_text())
        logger.on_epoch_end(epoch_idx, metrics={"loss": step_loss, "val_loss": val_loss, "accuracy": 0.0, "val_accuracy": 0.0})

logger.on_train_end(final_metrics={"loss": train_curve[-1], "val_loss": val_curve[-1]})
print("Total warnings:", len(warnings_engine.warnings))


In [ ]:
def generate_text(start="ROMEO:", length=400, temperature=0.9):
    model.eval()
    context = torch.tensor([stoi[c] for c in start], dtype=torch.long, device=device).unsqueeze(0)
    generated = list(start)
    hidden = None

    with torch.no_grad():
        for _ in range(length):
            logits, hidden = model(context[:, -context_len:], hidden)
            next_logits = logits[:, -1, :] / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            next_char = itos[int(next_idx.item())]
            generated.append(next_char)
            context = torch.cat([context, next_idx], dim=1)

    return "".join(generated)

sample = generate_text()
print(sample)

plt.figure(figsize=(8, 4))
plt.plot(train_curve, label="train")
plt.plot(val_curve, label="val")
plt.title("Character LM Loss")
plt.legend()
plt.show()
